# Власне донавчання: маршрутизація українських звернень
Користувач — оператор підтримки. Вхід — текст, вихід — одна з п’яти категорій. Класифікатор має стабільний формат, не потребує довгого промпту та дешевший за генеративну модель. RAG не навчає межі класів. DistilBERT multilingual — компактний базовий енкодер із новою головою. Дані синтетичні, створені для роботи, CC0; не містять персональних даних. Обмеження: шаблонні дані не гарантують якість на реальних зверненнях.

In [ ]:
%pip -q install "transformers>=4.48,<5" datasets accelerate scikit-learn

In [ ]:
import random,time,torch,numpy as np,pandas as pd,matplotlib.pyplot as plt
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding,set_seed
from datasets import Dataset
from sklearn.metrics import accuracy_score,f1_score,classification_report
set_seed(42)
assert torch.cuda.is_available(), "Увімкніть GPU T4"
print(torch.cuda.get_device_name())
labels=['Авторизація','Оплата','Технічна проблема','Налаштування','Пропозиція']
phrases=[['Не можу увійти','Забув пароль','Не надходить код входу','Акаунт заблоковано','Не працює двофакторний вхід','Сесія постійно завершується','Не приймає логін','Не можу відновити доступ','Пише неправильний пароль','Не пускає в особистий кабінет'],['Двічі списали кошти','Не проходить платіж','Потрібне повернення грошей','Не отримав квитанцію','Картку відхилено','Неправильна сума рахунку','Не можу оплатити підписку','Гроші списано без замовлення','Де мій рахунок','Не зараховано оплату'],['Сайт видає помилку 500','Сторінка не завантажується','Застосунок зависає','Кнопка не реагує','Файл не завантажується','Пошук не працює','Екран стає білим','Сервіс аварійно закривається','Зображення не відображаються','Запит завершується таймаутом'],['Як змінити електронну пошту','Хочу оновити номер телефону','Як вимкнути сповіщення','Де змінити мову','Як змінити аватар','Хочу налаштувати приватність','Як відредагувати ім’я','Де вибрати часовий пояс','Як увімкнути темну тему','Хочу змінити адресу профілю'],['Додайте експорт у PDF','Пропоную новий фільтр','Було б добре мати календар','Хочу функцію групового чату','Пропоную додати сортування','Зробіть інтеграцію з поштою','Додайте підтримку нових форматів','Хотілося б бачити статистику','Пропоную покращити меню','Додайте можливість створювати шаблони']]
wrappers=['{}','Вітаю! {}','Підкажіть, будь ласка: {}','Маю питання: {}','Допоможіть: {}','Добрий день, {}','Звертаюся щодо проблеми: {}','Потрібна допомога. {}','Прошу розглянути: {}','Буду вдячний за відповідь: {}']
rows={k:[] for k in ['train','validation','test']}
for label,items in enumerate(phrases):
 for i,text in enumerate(items):
  split='train' if i<6 else 'validation' if i<8 else 'test'
  for w in wrappers: rows[split].append({'text':w.format(text),'label':label})
for k,v in rows.items(): print(k,len(v))
assert len({r['text'] for v in rows.values() for r in v})==500
print(pd.DataFrame(rows['train']).sample(5,random_state=42))

## 1.2–1.3. Поділ і базовий рівень
300/100/100; різні основні формулювання в кожному наборі, щоб варіанти одного шаблону не перетиналися. Accuracy і macro F1 оцінюють загальну точність і рівномірність якості між класами. Базовий рівень — найчастіший клас (при рівності перший).

In [ ]:
checkpoint='distilbert/distilbert-base-multilingual-cased'
tok=AutoTokenizer.from_pretrained(checkpoint)
ds={k:Dataset.from_list(v).map(lambda b:tok(b['text'],truncation=True,max_length=96),batched=True) for k,v in rows.items()}
y=np.array(ds['test']['label']); majority=np.bincount(ds['train']['label']).argmax(); before=np.full(len(y),majority)
def metrics(y,p): return {'accuracy':accuracy_score(y,p),'f1':f1_score(y,p,average='macro',zero_division=0)}
base=metrics(y,before); print(base)
model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=5,id2label=dict(enumerate(labels)),label2id={v:i for i,v in enumerate(labels)})
def compute(e): return metrics(e.label_ids,e.predictions.argmax(-1))
args=TrainingArguments(output_dir='ft-output',num_train_epochs=3,learning_rate=3e-5,per_device_train_batch_size=16,per_device_eval_batch_size=32,eval_strategy='epoch',logging_steps=5,save_strategy='no',fp16=True,report_to='none',seed=42)
trainer=Trainer(model=model,args=args,train_dataset=ds['train'],eval_dataset=ds['validation'],data_collator=DataCollatorWithPadding(tok),compute_metrics=compute)

## 1.4. Донавчання
Повне донавчання енкодера і нової голови; LR 3e-5 для обережного оновлення ваг, 3 епохи, батч 16. LoRA не застосовується.

In [ ]:
torch.cuda.reset_peak_memory_stats(); start=time.perf_counter()
trainer.train(); elapsed=time.perf_counter()-start; peak=torch.cuda.max_memory_allocated()/1024**3
print(f'Час {elapsed:.1f} с; пікова GPU пам’ять {peak:.2f} GiB')
h=pd.DataFrame(trainer.state.log_history); display(h)
for col in ['loss','eval_loss']:
 if col in h: plt.plot(h.dropna(subset=[col])['step'],h.dropna(subset=[col])[col],marker='o',label=col)
plt.xlabel('Step');plt.ylabel('Loss');plt.legend();plt.show()

In [ ]:
pred=trainer.predict(ds['test']).predictions.argmax(-1); after=metrics(y,pred)
print(pd.DataFrame([base,after],index=['До: majority','Після: fine-tuning']))
print(classification_report(y,pred,target_names=labels,zero_division=0))
comparison=pd.DataFrame({'text':ds['test']['text'],'target':[labels[i] for i in y],'before':[labels[i] for i in before],'after':[labels[i] for i in pred]})
display(comparison.groupby('target').head(2)); errors=comparison[comparison.target!=comparison.after];display(errors.head(10))

## 1.6. Мій звіт
Задача: автоматична маршрутизація звернень. Модель і дані: multilingual DistilBERT, 500 синтетичних прикладів. Метрики: accuracy та macro F1. Метод: повне донавчання з новою головою. Фактичні результати формуються нижче. Аналіз конкретних помилок і перенавчання необхідно уточнити після запуску.

In [ ]:
print(f"До: accuracy={base['accuracy']:.3f}, F1={base['f1']:.3f}; після: accuracy={after['accuracy']:.3f}, F1={after['f1']:.3f}.")
print(f"Тренування: {elapsed:.1f} с, peak {peak:.2f} GiB. Помилок: {len(errors)} з {len(y)}.")
print('Висновок:', 'Донавчання покращило accuracy на цьому наборі.' if after['accuracy']>base['accuracy'] else 'Покращення accuracy не підтверджено.')
print('Обмеження: синтетичні дані; потрібна перевірка на реальних зверненнях.')